In [1]:
import time

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
from torchvision import transforms, datasets

# 一.训练与测试函数包装

In [2]:
def train(dataloader, model, loss_fn, optimizer):
    train_loss, correct = 0, 0

    for i, (image, label) in enumerate(dataloader):
        image = image.view(image.size(0), -1)
        pred = model(image)
        loss = loss_fn(pred, label)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        with torch.no_grad():
            correct += (pred.argmax(1) == label).sum().item()
            train_loss += loss.item()

    train_loss /= len(dataloader)
    correct /= len(dataloader.dataset)

    return train_loss, correct

In [3]:
def test(dataloader, model, loss_fn):
    test_loss, correct = 0, 0

    for image, label in dataloader:
        image = image.view(image.size(0), -1)
        pred = model(image)
        loss = loss_fn(pred, label)

        correct += (pred.argmax(1) == label).sum().item()
        test_loss += loss.item()

    test_loss /= len(dataloader)
    correct /= len(dataloader.dataset)
    return test_loss, correct

# 二.定义模型

In [4]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(784, 256)
        self.act1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 64)
        self.act2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 10)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act1(x)
        x = self.fc2(x)
        x = self.act2(x)
        x = self.fc3(x)
        return x

model = MLP()

# 三.加载数据

In [5]:
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transforms.ToTensor())
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transforms.ToTensor())

In [6]:
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# 四. 损失函数与优化器

In [7]:
loss_fn = nn.CrossEntropyLoss(reduction='sum')

In [8]:
optimizer = optim.SGD(model.parameters(), lr=0.001)

# 五.训练

In [9]:
print(f"| {'':^6s} | {'time':^15s} | {'loss':^15s} | {'acc':^15s} |")
print(f"| {'epoch':^6s} | {'train':^6s} | {'test':^6s} | {'train':^6s} | {'test':^6s} | {'train':^6s} | {'test':^6s} |")

for epoch in range(50):

    time1 = time.time()
    train_loss, train_correct = train(train_dataloader, model, loss_fn, optimizer)
    time2 = time.time()
    test_loss, test_correct = test(test_dataloader, model, loss_fn)
    time3 = time.time()

    print(f"| {epoch:6d} | {time2-time1:.4f} | {time3-time2:.4f} | {train_loss:.4f} | {test_loss:.4f} | {train_correct:.4f} | {test_correct:.4f} |")


|        |      time       |      loss       |       acc       |
| epoch  | train  |  test  | train  |  test  | train  |  test  |
|      0 | 6.3271 | 0.8307 | 9.9079 | 4.2035 | 0.8267 | 0.9241 |
|      1 | 6.3574 | 0.8292 | 3.7563 | 2.9763 | 0.9318 | 0.9456 |
|      2 | 6.3589 | 0.8294 | 2.7091 | 2.3493 | 0.9507 | 0.9577 |
|      3 | 6.3604 | 0.8284 | 2.0681 | 1.9220 | 0.9622 | 0.9628 |
|      4 | 6.3577 | 0.8288 | 1.6636 | 1.6873 | 0.9695 | 0.9666 |
|      5 | 6.3563 | 0.8301 | 1.3729 | 1.4509 | 0.9749 | 0.9718 |
|      6 | 6.3520 | 0.8291 | 1.1627 | 1.4120 | 0.9785 | 0.9723 |
|      7 | 6.4328 | 0.8511 | 0.9788 | 1.3667 | 0.9823 | 0.9738 |
|      8 | 6.5108 | 0.8499 | 0.8462 | 1.1978 | 0.9846 | 0.9770 |
|      9 | 6.5161 | 0.8506 | 0.7300 | 1.0907 | 0.9868 | 0.9772 |
|     10 | 6.5170 | 0.8497 | 0.6305 | 1.1146 | 0.9888 | 0.9778 |
|     11 | 6.5143 | 0.8491 | 0.5456 | 1.1584 | 0.9904 | 0.9773 |
|     12 | 6.5205 | 0.8488 | 0.4773 | 1.1067 | 0.9917 | 0.9780 |
|     13 | 2.7343 | 0.297